In [7]:
from datasets import load_dataset
from vllm import LLM, SamplingParams
import json
from openai import OpenAI
from langchain_openai import ChatOpenAI

from textwrap import dedent
ds1 = load_dataset("json", data_files="/mnt/data1tb/thangcn/datnv2/data/qa_documents/test.json", split="train")
# ds2 = load_dataset("csv", data_files="/mnt/data1tb/thangcn/datnv2/data/qa_documents/eval_predictions.csv", split="train")


In [2]:
ds1

Dataset({
    features: ['question', 'context', 'answer', 'text', 'token_count'],
    num_rows: 100
})

In [3]:
df1 = ds1.to_pandas()
df1 = df1.drop(columns=['token_count'])
df1.text.iloc[0]

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\nUse only the information to answer the question<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nTại sao nướu răng của bé lại nhô lên?\n\nInformation:\n\n```\nảnh do bạn đọc cung cấpchào bạn,lợi sẽ đi theo phần xương bên dưới, mà xương sẽ phát triển dựa theo răng. do hai răng cửa của bé quá hô và chìa ra phía trước nên phần xương ổ răng bao quanh chân răng cũng sẽ phát triển về phía trước. khi phần xương ổ nằm hẳn về phía trước như vậy thì tất nhiên nướu răng/lợi cũng sẽ nhô lên như trường hợp của con bạn. tốt nhất bạn nên đưa bé đi chỉnh hình răng sớm để cải thiện không chỉ về thẩm mỹ mà còn chức năng ăn nhai cũng như phát âm cho bé, bạn nhé! thân mến, alobacsi.comcổng thông tin tư vấn sức khỏe miễn phí\n```<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nDo hai răng cửa của bé quá hô và chìa ra phía trước nên phần xương ổ răng bao quanh chân 

In [4]:
df1

,question,context,answer,text
0,Tại sao nướu răng của bé lại nhô lên?,"ảnh do bạn đọc cung cấpchào bạn,lợi sẽ đi theo...",Do hai răng cửa của bé quá hô và chìa ra phía ...,<|begin_of_text|><|start_header_id|>system<|en...
1,Ngưng isoniazid 6 ngày liệu có gây kháng thuốc?,"- chào em,cả 2 thuốc trên đều có thành phần ch...","Không, ngưng isoniazid 6 ngày không gây kháng ...",<|begin_of_text|><|start_header_id|>system<|en...
2,Bố mẹ nên làm gì khi bé dùng thuốc thoa trên v...,"- chào em,bây giờ, em nên ngưng cho bé dùng th...",Nên ngưng cho bé dùng thuốc và đưa bé đi khám ...,<|begin_of_text|><|start_header_id|>system<|en...
3,Mã tương đương của dịch vụ phẫu thuật nội soi ...,"Mã tương đương: 03.4068.0451, Tên dịch vụ kỹ t...",03.4068.0451,<|begin_of_text|><|start_header_id|>system<|en...
4,Bệnh tiểu máu có thể được chẩn đoán như thế nào?,"chào bạn,bạn không mô tả rõ những triệu chứng ...",Chẩn đoán tiểu máu cần có xét nghiệm tìm hồng ...,<|begin_of_text|><|start_header_id|>system<|en...
...,...,...,...,...
95,"Trước khi xét nghiệm HBSAg, tôi cần xác định l...","- chào em vân,trước khi câu hỏi của em, tôi cầ...",Tốt nhất là kiểm tra lại để chắc chắn. Nếu kết...,<|begin_of_text|><|start_header_id|>system<|en...
96,Tại sao xịt rửa mũi quá mạnh có thể gây viêm t...,khi xịt rửa mũi quá mạnh dịch mũi xâm nhập vào...,Dịch mũi xâm nhập vào tai giữa qua vòi nhĩ (eu...,<|begin_of_text|><|start_header_id|>system<|en...
97,Vi rút HPV lây lan qua đường nào?,"chào bạn, bạn cung cấp chưa có dấu hiệu nghi n...","Quan hệ tình dục qua đường âm đạo, hậu môn, đư...",<|begin_of_text|><|start_header_id|>system<|en...
98,Em không ghi rõ huyết áp của em thấp là thấp b...,"chào thu,em không ghi rõ huyết áp của em thấp ...",Theo mô tả thì vấn đề của em nằm nhiều ở nguyê...,<|begin_of_text|><|start_header_id|>system<|en...


In [5]:
llm = ChatOpenAI(
    base_url="http://localhost:8000/v1",
    api_key="dummy",
    temperature=0.01,
    model="Qwen/Qwen3-8B",
)

In [8]:
def process_llm_function_call(messages):
    response = llm.predict_messages(
        messages,
    )

    return response

In [ ]:
def create_test_prompt(data_row):
    prompt = dedent(
        f"""
    {data_row["question"]}

    Information:

    ```
    {data_row["context"]}
    ```
    """
    )
    messages = [
        {
            "role": "system",
            "content": "You are an expert in the medical field. Use the information only to answer the medical question. Please answer in Vietnamese for me.",
        },
        {"role": "user", "content": prompt},
    ]
    return messages

# def create_test_prompt(data_row):
#     prompt = dedent(
#         f"""
#     {data_row["question"]}

#     Information:

#     ```
#     {data_row["context"]}
#     ```
#     """
#     )
#     messages = [
#         {
#             "role": "system",
#             "content": "Use the information only to answer the question. Please answer in Vietnamese for me.",
#         },
#         {"role": "user", "content": prompt},
#     ]
#     return tokenizer.apply_chat_template(
#         messages, tokenize=False, add_generation_prompt=True
#     )

In [ ]:
# from transformers import (
#     AutoModelForCausalLM,
#     AutoTokenizer,
#     BitsAndBytesConfig,
#     pipeline,
# )
# from textwrap import dedent
# import torch

In [ ]:
# from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
# import torch

# # Cấu hình quantization 8-bit
# quantization_config = BitsAndBytesConfig(
#     load_in_8bit=True,
#     llm_int8_threshold=6.0,
#     llm_int8_skip_modules=None,  # Bỏ qua module nào không quantize (nếu cần)
# )

# # Load model với quantization
# model = AutoModelForCausalLM.from_pretrained(
#     "Qwen/Qwen3-8B",
#     device_map="auto",  # Tự động chọn GPU/CPU
#     quantization_config=quantization_config,
#     torch_dtype=torch.float16,  # Kết hợp với float16 để tiết kiệm thêm bộ nhớ
# )

# tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B")

Loading checkpoint shards: 100%|██████████| 5/5 [00:10<00:00,  2.05s/it]


In [10]:
row = df1.iloc[0]
prompt = create_test_prompt(row)

In [11]:
prompt

[{'role': 'system',
  'content': 'You are an expert in the medical field. Use the information only to answer the medical question. Please answer in Vietnamese for me.'},
 {'role': 'user',
  'content': '\nTại sao nướu răng của bé lại nhô lên?\n\nInformation:\n\n```\nảnh do bạn đọc cung cấpchào bạn,lợi sẽ đi theo phần xương bên dưới, mà xương sẽ phát triển dựa theo răng. do hai răng cửa của bé quá hô và chìa ra phía trước nên phần xương ổ răng bao quanh chân răng cũng sẽ phát triển về phía trước. khi phần xương ổ nằm hẳn về phía trước như vậy thì tất nhiên nướu răng/lợi cũng sẽ nhô lên như trường hợp của con bạn. tốt nhất bạn nên đưa bé đi chỉnh hình răng sớm để cải thiện không chỉ về thẩm mỹ mà còn chức năng ăn nhai cũng như phát âm cho bé, bạn nhé! thân mến, alobacsi.comcổng thông tin tư vấn sức khỏe miễn phí\n```\n'}]

In [13]:
process_llm_function_call(prompt).content

'<think>\nOkay, let\'s tackle this question. The user is asking why a baby\'s gums are protruding. The information provided mentions that the gums (lợi) follow the bone structure, and if the baby\'s front teeth are protruding (hô), the bone around the tooth roots develops forward. As a result, the gums also protrude. The advice is to consult a dentist early for orthodontic treatment to improve aesthetics, function, and speech.\n\nFirst, I need to make sure I understand the medical terms here. "Hô" in Vietnamese refers to an overjet, which is when the upper front teeth protrude beyond the lower ones. The explanation given is that the bone around the teeth (xương ổ răng) develops forward because the teeth are too far forward, causing the gums to bulge. \n\nI should verify if this is a common cause. In orthodontics, when teeth are malpositioned, the supporting bone can indeed remodel. If the upper front teeth are significantly protruding, the alveolar bone (which holds the teeth) might gr

In [ ]:
# pipe = pipeline(
#     "text-generation",
#     model=model,
#     tokenizer=tokenizer,
#     do_sample=True,
#     temperature=0.01,     # > 0.0
#     top_p=0.9,
#     top_k=50,
#     pad_token_id=tokenizer.eos_token_id,
#     return_full_text=False
# )
# pipe(prompt)

Device set to use cuda:0


In [14]:
from tqdm import tqdm

In [16]:
import re
from tqdm import tqdm

def extract_answer(text):
    match = re.search(r'</think>\s*\n*(.+)', text, re.DOTALL)
    if match:
        return match.group(1).strip()
    return "Không tìm thấy câu trả lời sau thẻ </think>"

predictions = []
for index, row in tqdm(df1.iterrows(), total=df1.shape[0]):
    outputs = process_llm_function_call(create_test_prompt(row)).content
    # Trích xuất câu trả lời sau thẻ </think>
    answer = extract_answer(outputs)
    predictions.append(answer)


100%|██████████| 100/100 [15:05<00:00,  9.06s/it]


In [17]:
df1['response'] = predictions

In [18]:
df1['response'].iloc[1]

'Ngưng sử dụng isoniazid trong 6 ngày **có thể làm tăng nguy cơ kháng thuốc**, nhưng mức độ phụ thuộc vào nhiều yếu tố như tình trạng bệnh, liều lượng, và liệu pháp điều trị tổng thể. Dưới đây là phân tích chi tiết:\n\n1. **Nguy cơ kháng thuốc**:  \n   - Isoniazid là một trong những thuốc chống lao đầu tiên, thường được dùng kết hợp với các thuốc khác (như rifampicin, pyrazinamid, ethambutol) để ngăn ngừa kháng thuốc.  \n   - Nếu ngưng thuốc trong thời gian ngắn (6 ngày), vi khuẩn lao có thể **không đủ thời gian phát triển kháng thuốc**, nhưng việc gián đoạn điều trị có thể làm **giảm hiệu quả của liệu pháp**, dẫn đến nguy cơ tái phát hoặc kháng thuốc trong tương lai.  \n\n2. **Tác động của việc ngưng thuốc**:  \n   - **Ngừng thuốc đột ngột** có thể làm cho vi khuẩn lao còn sót lại trong cơ thể có cơ hội sinh sôi và phát triển kháng thuốc, đặc biệt nếu không được điều trị tiếp.  \n   - Trong trường hợp **kháng thuốc đã tồn tại** (ví dụ: vi khuẩn kháng isoniazid), việc ngưng thuốc sẽ là

In [19]:
df1['reference'] = df1['answer']
df1 = df1.drop(columns=['answer', 'text'])
df1

,question,context,response,reference
0,Tại sao nướu răng của bé lại nhô lên?,"ảnh do bạn đọc cung cấpchào bạn,lợi sẽ đi theo...",Nướu răng của bé nhô lên có thể do **răng cửa ...,Do hai răng cửa của bé quá hô và chìa ra phía ...
1,Ngưng isoniazid 6 ngày liệu có gây kháng thuốc?,"- chào em,cả 2 thuốc trên đều có thành phần ch...",Ngưng sử dụng isoniazid trong 6 ngày **có thể ...,"Không, ngưng isoniazid 6 ngày không gây kháng ..."
2,Bố mẹ nên làm gì khi bé dùng thuốc thoa trên v...,"- chào em,bây giờ, em nên ngưng cho bé dùng th...",Khi bé dùng thuốc thoa có thể gây tác dụng phụ...,Nên ngưng cho bé dùng thuốc và đưa bé đi khám ...
3,Mã tương đương của dịch vụ phẫu thuật nội soi ...,"Mã tương đương: 03.4068.0451, Tên dịch vụ kỹ t...",Mã tương đương của dịch vụ phẫu thuật nội soi ...,03.4068.0451
4,Bệnh tiểu máu có thể được chẩn đoán như thế nào?,"chào bạn,bạn không mô tả rõ những triệu chứng ...",Bệnh tiểu máu (tiểu ra máu) được chẩn đoán thô...,Chẩn đoán tiểu máu cần có xét nghiệm tìm hồng ...
...,...,...,...,...
95,"Trước khi xét nghiệm HBSAg, tôi cần xác định l...","- chào em vân,trước khi câu hỏi của em, tôi cầ...","Trước khi xét nghiệm HBsAg, bạn cần xác định r...",Tốt nhất là kiểm tra lại để chắc chắn. Nếu kết...
96,Tại sao xịt rửa mũi quá mạnh có thể gây viêm t...,khi xịt rửa mũi quá mạnh dịch mũi xâm nhập vào...,Xịt rửa mũi quá mạnh có thể gây viêm tai giữa ...,Dịch mũi xâm nhập vào tai giữa qua vòi nhĩ (eu...
97,Vi rút HPV lây lan qua đường nào?,"chào bạn, bạn cung cấp chưa có dấu hiệu nghi n...",Vi rút HPV (human papillomavirus) lây lan chủ ...,"Quan hệ tình dục qua đường âm đạo, hậu môn, đư..."
98,Em không ghi rõ huyết áp của em thấp là thấp b...,"chào thu,em không ghi rõ huyết áp của em thấp ...","Chào bạn, \nCảm ơn bạn đã chia sẻ thông tin. ...",Theo mô tả thì vấn đề của em nằm nhiều ở nguyê...


In [20]:
import os
from dotenv import load_dotenv

In [21]:
load_dotenv('/mnt/data1tb/thangcn/datnv2/.env')
open_ai_key = os.getenv("OPENAI_API_KEY")
# groq_api_key = os.getenv("GROQ_API_KEY")
MODEL = 'gpt-4o' #os.getenv("MODEL", "gpt-4o")

llm = ChatOpenAI(model=MODEL, temperature=0.3, api_key=open_ai_key)

In [22]:
df1['retrieved_contexts'] = df1['context']

In [23]:
for i in range(len(df1['retrieved_contexts'])):
    df1['retrieved_contexts'].iloc[i] = [df1['retrieved_contexts'].iloc[i]]
    

/tmp/ipykernel_514465/3607375772.py:2: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df1['retrieved_contexts'].iloc[i] = [df1['retrieved_contexts'].iloc[i]]
/tmp/ipykernel_514465/3607375772.py:2: FutureWarning: ChainedAssignmentError: behavi

In [24]:
df1['retrieved_contexts'].iloc[0]

['ảnh do bạn đọc cung cấpchào bạn,lợi sẽ đi theo phần xương bên dưới, mà xương sẽ phát triển dựa theo răng. do hai răng cửa của bé quá hô và chìa ra phía trước nên phần xương ổ răng bao quanh chân răng cũng sẽ phát triển về phía trước. khi phần xương ổ nằm hẳn về phía trước như vậy thì tất nhiên nướu răng/lợi cũng sẽ nhô lên như trường hợp của con bạn. tốt nhất bạn nên đưa bé đi chỉnh hình răng sớm để cải thiện không chỉ về thẩm mỹ mà còn chức năng ăn nhai cũng như phát âm cho bé, bạn nhé! thân mến, alobacsi.comcổng thông tin tư vấn sức khỏe miễn phí']

In [25]:
from datasets import Dataset

ds = Dataset.from_pandas(df1)

In [26]:
ds

Dataset({
    features: ['question', 'context', 'response', 'reference', 'retrieved_contexts'],
    num_rows: 100
})

In [27]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [28]:
from ragas.metrics import answer_correctness, faithfulness, answer_relevancy
from ragas import evaluate

In [29]:
EMBED_MODEL = "thang1943/multilingual-e5-large-v2" #os.getenv("EMBED_MODEL", "nampham1106/bkcare-embedding")

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={'device': 'cpu'}
)


/tmp/ipykernel_514465/400226938.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [30]:
ragas_results = evaluate(ds, metrics=[answer_correctness, answer_relevancy, faithfulness], llm = llm, embeddings=embeddings)

Evaluating:   0%|          | 0/300 [00:00<?, ?it/s]

Evaluating: 100%|██████████| 300/300 [02:34<00:00,  1.94it/s]


In [31]:
ragas_results

{'answer_correctness': 0.4394, 'answer_relevancy': 0.5600, 'faithfulness': 0.6253}

In [62]:
import tiktoken 

In [66]:
def count_tokens(text, model="gpt-4o"):
    encoder = tiktoken.encoding_for_model(model)
    return len(encoder.encode(text))

# --- Tính token trước khi chạy đánh giá ---
def estimate_ragas_usage(dataset):
    total_tokens = 0
    prompt_template = """
    Evaluate if the answer directly addresses the question (1/0):
    Question: {question}
    Answer: {answer}
    """
    prompt_tokens = count_tokens(prompt_template)

    for q, c, r1, r2 in zip(dataset["question"], dataset["context"], dataset["response"], dataset["reference"]):
        question_tokens = count_tokens(q)
        context_tokens = count_tokens(c)
        response_tokens = count_tokens(r1)
        reference_tokens = count_tokens(r2)

        total_tokens += (prompt_tokens + question_tokens + context_tokens + response_tokens + reference_tokens)
    
    return total_tokens

# --- Ước lượng token ---
estimated_tokens = estimate_ragas_usage(ds)
print(f"Estimated tokens needed: {estimated_tokens}")

Estimated tokens needed: 30321
